In [ ]:
# Lee-Mykland Jump Test (see for exact derivation and explain choice of K etc.)

def lm_test(df: pd.DataFrame, K: int = 270, significance_level: float = 0.05):

    """
    Assumptions
        - df is resampled and missing timestamps due to maintenances, halts, weekends etc. are already filtered out
        - df contains closing prices (i.e. OHLCV format), not returns
        - K depends on length of trading day
    """

    df = df.copy()

    # Input validation
    if not df.index.is_monotonic_increasing:
        df = df.sort_index(ascending=True)

    # Input validation
    if not 'close' in df.columns:
        raise ValueError('Dataframe does not contain close series')

    if K < 2 or len(df) <= K + 1:
        raise ValueError('Insufficient observations for K')

    if not 0 < significance_level < 1:
        raise ValueError('Significance level must be within 0-1 range')

    n = len(df['close'])
    c = (2/np.pi)**0.5
    Sn = 1/(c*(2*np.log(n))**0.5)
    Cn = ((2*np.log(n))**0.5)/c - (np.log(np.pi) + np.log(np.log(n)))/(2*c*(2*np.log(n))**0.5)
    beta_star = -np.log(-np.log(1-significance_level)) # Quantile for critical value (H1 threshold) based on Gumbel distribution (inverse CDF)

    # Log-returns
    df['log_return'] = np.log(df['close']) - np.log(df['close'].shift(1))

    # Initialize volatility column
    df['sigma_hat'] = np.nan

    # Bipower-Variation
    for i in range(K+1, len(df)):

        bipower_var = 0.0

        for j in range(i-K+2, i+1):

            bipower_var += abs(df['log_return'].iloc[j-1])*abs(df['log_return'].iloc[j-2])

        bipower_var /= (K-1)
        sigma_hat = np.sqrt((np.pi / 2) * bipower_var)

        df.loc[df.index[i], 'sigma_hat'] = sigma_hat

    # Test-statistic
    df['L'] = df['log_return']/df['sigma_hat']

    # Standardized test-statistic
    df['T'] = (np.abs(df['L']) - Cn) / Sn

    # Binary (boolean) check for jump
    df['J'] = (df['T'] > beta_star).fillna(False).astype(int)

    # Signed jumps
    df['J'] = df['J']*np.sign(df['log_return']).fillna(0).astype(int)

    return df